# Week 3 - Data Preprocessing

This notebook prepares residential single-family sales for modeling. It handles missing values, encodes categorical fields, scales numerical fields, and chooses a rolling training-window length without using the final test month.

## Setup and feature choices

Only property characteristics and location fields are used. `ClosePrice` is the target. Transaction identifiers, agent information, and post-close fields are excluded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

DATA_DIR = Path("data")
CSV_FILES = sorted(DATA_DIR.glob("CRMLSSold*.csv"))
CLEANED_PATH = DATA_DIR / "week3_cleaned.csv"

NUMERIC_FEATURES = [
    "LivingArea", "Bedrooms", "Bathrooms", "LotSizeSquareFeet",
    "YearBuilt", "Stories", "GarageSpaces", "ParkingTotal",
    "Latitude", "Longitude",
]
CATEGORICAL_FEATURES = ["City", "County", "PostalCode", "Levels"]
BINARY_FEATURES = [
    "AttachedGarage", "PrivatePool", "View", "Waterfront",
    "NewConstruction", "Fireplace",
]

len(CSV_FILES), CSV_FILES[0].name, CSV_FILES[-1].name

(30, 'CRMLSSold20220101_20231231_filled.csv', 'CRMLSSold202605.csv')

## Load and clean the modeling population

Rows without a usable close date or positive target are dropped because neither the time split nor supervised training is possible for them. Invalid structural values are changed to missing and handled by the training-fitted preprocessor.

In [2]:
RAW_COLUMNS = [
    "CloseDate", "ClosePrice", "PropertyType", "PropertySubType",
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "LotSizeAcres", "YearBuilt", "Stories",
    "GarageSpaces", "ParkingTotal", "FireplacesTotal", "Latitude",
    "Longitude", "City", "CountyOrParish", "PostalCode", "Levels",
    "AttachedGarageYN", "PoolPrivateYN", "ViewYN", "WaterfrontYN",
    "NewConstructionYN", "FireplaceYN",
]

frames = []
for path in CSV_FILES:
    header = pd.read_csv(path, nrows=0).columns
    available = [column for column in RAW_COLUMNS if column in header]
    frame = pd.read_csv(path, usecols=available, low_memory=False)
    for column in RAW_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
    frames.append(frame[RAW_COLUMNS])

raw = pd.concat(frames, ignore_index=True)
raw["RecordId"] = np.arange(len(raw), dtype=np.int64)

property_type = raw["PropertyType"].astype("string").str.strip()
property_subtype = raw["PropertySubType"].astype("string").str.strip()
raw = raw.loc[
    property_type.eq("Residential")
    & property_subtype.eq("SingleFamilyResidence")
].copy()

def to_number(series):
    return pd.to_numeric(
        series.astype("string").str.replace(r"[$,]", "", regex=True),
        errors="coerce",
    )

def to_binary(series):
    normalized = series.astype("string").str.strip().str.lower()
    return normalized.map({"true": 1.0, "yes": 1.0, "y": 1.0, "1": 1.0,
                           "false": 0.0, "no": 0.0, "n": 0.0, "0": 0.0})

model_data = pd.DataFrame(index=raw.index)
model_data["RecordId"] = raw["RecordId"]
model_data["CloseDate"] = pd.to_datetime(raw["CloseDate"], errors="coerce")
model_data["ClosePrice"] = to_number(raw["ClosePrice"])
model_data["LivingArea"] = to_number(raw["LivingArea"])
model_data["Bedrooms"] = to_number(raw["BedroomsTotal"])
model_data["Bathrooms"] = to_number(raw["BathroomsTotalInteger"])
lot_square_feet = to_number(raw["LotSizeSquareFeet"])
lot_acres = to_number(raw["LotSizeAcres"]) * 43_560
model_data["LotSizeSquareFeet"] = lot_square_feet.combine_first(lot_acres)
for column in ["YearBuilt", "Stories", "GarageSpaces", "ParkingTotal",
               "FireplacesTotal", "Latitude", "Longitude"]:
    model_data[column] = to_number(raw[column])

category_map = {"City": "City", "County": "CountyOrParish",
                "PostalCode": "PostalCode", "Levels": "Levels"}
for output, source in category_map.items():
    values = raw[source].astype("string").str.strip().replace("", pd.NA)
    if output == "PostalCode":
        values = values.str.replace(r"\.0$", "", regex=True)
    model_data[output] = values

binary_map = {"AttachedGarage": "AttachedGarageYN", "PrivatePool": "PoolPrivateYN",
              "View": "ViewYN", "Waterfront": "WaterfrontYN",
              "NewConstruction": "NewConstructionYN", "Fireplace": "FireplaceYN"}
for output, source in binary_map.items():
    model_data[output] = to_binary(raw[source])

model_data.loc[model_data["LivingArea"] <= 0, "LivingArea"] = np.nan
model_data.loc[model_data["LotSizeSquareFeet"] <= 0, "LotSizeSquareFeet"] = np.nan
model_data.loc[~model_data["YearBuilt"].between(1700, 2026), "YearBuilt"] = np.nan
model_data.loc[~model_data["Latitude"].between(32, 42), "Latitude"] = np.nan
model_data.loc[~model_data["Longitude"].between(-125, -114), "Longitude"] = np.nan
for column in ["Bedrooms", "Bathrooms", "Stories", "GarageSpaces",
               "ParkingTotal", "FireplacesTotal"]:
    model_data.loc[model_data[column] < 0, column] = np.nan

before_required_drop = len(model_data)
model_data = model_data.loc[
    model_data["CloseDate"].notna() & (model_data["ClosePrice"] > 0)
].copy()
model_data["CloseMonth"] = model_data["CloseDate"].dt.to_period("M")

pd.Series({"rows_in_scope": before_required_drop, "usable_rows": len(model_data),
           "dropped_missing_date_or_target": before_required_drop - len(model_data),
           "first_month": str(model_data["CloseMonth"].min()),
           "last_month": str(model_data["CloseMonth"].max())})

rows_in_scope                      399157
usable_rows                        399154
dropped_missing_date_or_target          3
first_month                       2022-01
last_month                        2026-05
dtype: object

## Missing-value, encoding, and scaling policy

The preprocessor is fitted on training data only. Numeric fields are clipped at training 1st/99th percentiles, median-imputed, and standardized. Binary fields use the training mode. High-cardinality categories use training-frequency encoding, with unseen values mapped to zero. Missing indicators preserve information about every imputation.

In [3]:
def fit_preprocessor(train):
    state = {"numeric": {}, "binary": {}, "categorical": {}}

    for column in NUMERIC_FEATURES:
        values = train[column].astype(float)
        lower, upper = values.quantile([0.01, 0.99])
        if pd.isna(lower) or pd.isna(upper):
            lower, upper = 0.0, 0.0
        clipped = values.clip(lower, upper)
        median = clipped.median()
        median = 0.0 if pd.isna(median) else float(median)
        filled = clipped.fillna(median)
        mean = float(filled.mean())
        scale = float(filled.std(ddof=0))
        if not np.isfinite(scale) or scale == 0:
            scale = 1.0
        state["numeric"][column] = {"lower": float(lower), "upper": float(upper),
                                      "median": median, "mean": mean, "scale": scale}

    for column in BINARY_FEATURES:
        mode = train[column].mode(dropna=True)
        # Waterfront is sparsely reported: absent values are treated as non-waterfront,
        # while the separate missing flag preserves that the source value was absent.
        state["binary"][column] = (
            0.0 if column == "Waterfront" else float(mode.iloc[0]) if len(mode) else 0.0
        )

    for column in CATEGORICAL_FEATURES:
        values = train[column].fillna("__MISSING__").astype(str)
        state["categorical"][column] = values.value_counts(normalize=True).to_dict()

    return state

def transform_features(data, state):
    transformed = pd.DataFrame(index=data.index)

    for column in NUMERIC_FEATURES:
        settings = state["numeric"][column]
        values = data[column].astype(float)
        transformed[f"{column}_missing"] = values.isna().astype("int8")
        values = values.clip(settings["lower"], settings["upper"]).fillna(settings["median"])
        transformed[f"{column}_scaled"] = (values - settings["mean"]) / settings["scale"]

    for column in BINARY_FEATURES:
        values = data[column].astype(float)
        transformed[f"{column}_missing"] = values.isna().astype("int8")
        transformed[column] = values.fillna(state["binary"][column]).astype("int8")

    for column in CATEGORICAL_FEATURES:
        values = data[column].fillna("__MISSING__").astype(str)
        transformed[f"{column}_missing"] = data[column].isna().astype("int8")
        transformed[f"{column}_frequency"] = (
            values.map(state["categorical"][column]).fillna(0.0).astype(float)
        )

    return transformed

missing_summary = pd.DataFrame({
    "missing_rows": model_data[NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES].isna().sum(),
    "missing_percent": model_data[NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES].isna().mean() * 100,
}).sort_values("missing_percent", ascending=False)
missing_summary

,missing_rows,missing_percent
Waterfront,398979,99.9562
Stories,54596,13.6779
AttachedGarage,46109,11.5517
Levels,41689,10.4443
PrivatePool,40547,10.1582
NewConstruction,39538,9.9055
View,39517,9.9002
GarageSpaces,14597,3.6570
ParkingTotal,7645,1.9153
LotSizeSquareFeet,7015,1.7575


## Tune the number of training months

The latest month is reserved as the final test set. The month immediately before it is validation data for choosing `X`. Candidate models use log price and a small linear baseline; median absolute percentage error is the selection metric because it is resistant to a few implausible sale-price outliers. The validation month is not included when fitting preprocessing statistics or model coefficients.

In [4]:
def fit_linear_log_model(features, target):
    matrix = np.column_stack([np.ones(len(features)), features.to_numpy(dtype=float)])
    coefficients, *_ = np.linalg.lstsq(matrix, np.log1p(target.to_numpy(dtype=float)), rcond=None)
    return coefficients

def predict_linear_log_model(features, coefficients):
    matrix = np.column_stack([np.ones(len(features)), features.to_numpy(dtype=float)])
    log_prediction = np.clip(matrix @ coefficients, 0, np.log1p(100_000_000))
    return np.expm1(log_prediction)

test_month = model_data["CloseMonth"].max()
validation_month = test_month - 1
candidate_windows = [3, 6, 12, 18, 24, 36, 48]
validation = model_data.loc[model_data["CloseMonth"] == validation_month]
window_results = []

for months in candidate_windows:
    start_month = validation_month - months
    candidate_train = model_data.loc[
        model_data["CloseMonth"].between(start_month, validation_month - 1)
    ]
    if candidate_train["CloseMonth"].nunique() < months:
        continue

    candidate_state = fit_preprocessor(candidate_train)
    train_features = transform_features(candidate_train, candidate_state)
    validation_features = transform_features(validation, candidate_state)
    coefficients = fit_linear_log_model(train_features, candidate_train["ClosePrice"])
    predictions = predict_linear_log_model(validation_features, coefficients)
    actual = validation["ClosePrice"].to_numpy(dtype=float)
    absolute_errors = np.abs(actual - predictions)

    window_results.append({
        "training_months": months,
        "training_rows": len(candidate_train),
        "validation_month": str(validation_month),
        "validation_rows": len(validation),
        "validation_MAE_dollars": absolute_errors.mean(),
        "validation_MdAPE_percent": np.median(absolute_errors / actual) * 100,
    })

window_results = pd.DataFrame(window_results).sort_values("training_months").reset_index(drop=True)
selected_x = int(window_results.loc[window_results["validation_MdAPE_percent"].idxmin(), "training_months"])
window_results.style.format({"validation_MAE_dollars": "${:,.0f}",
                             "validation_MdAPE_percent": "{:.2f}%"})

,training_months,training_rows,validation_month,validation_rows,validation_MAE_dollars,validation_MdAPE_percent
0,3,27217,2026-04,12031,"$610,092",22.24%
1,6,59440,2026-04,12031,"$609,924",22.15%
2,12,129821,2026-04,12031,"$609,064",22.14%
3,18,191088,2026-04,12031,"$608,730",22.14%
4,24,266928,2026-04,12031,"$608,395",22.22%
5,36,369527,2026-04,12031,"$609,082",22.04%
6,48,370686,2026-04,12031,"$609,188",22.04%


In [5]:
pd.Series({"selected_training_months_X": selected_x,
           "selection_metric": "lowest validation MdAPE",
           "validation_month": str(validation_month),
           "untouched_test_month": str(test_month)})

selected_training_months_X                         36
selection_metric              lowest validation MdAPE
validation_month                              2026-04
untouched_test_month                          2026-05
dtype: object

## Fit final preprocessing and export

After selecting `X`, the training window rolls forward by one month so it ends immediately before the test month. Preprocessing is refitted on that final training window and then applied unchanged to the test rows.

In [6]:
final_train_start = test_month - selected_x
final_train = model_data.loc[
    model_data["CloseMonth"].between(final_train_start, test_month - 1)
].copy()
final_test = model_data.loc[model_data["CloseMonth"] == test_month].copy()

final_state = fit_preprocessor(final_train)
train_features = transform_features(final_train, final_state)
test_features = transform_features(final_test, final_state)

def assemble_cleaned(source, features, split):
    metadata = source[["RecordId", "CloseDate", "CloseMonth", "ClosePrice"]].copy()
    metadata["CloseMonth"] = metadata["CloseMonth"].astype(str)
    metadata["split"] = split
    return pd.concat([metadata, features], axis=1)

cleaned = pd.concat([
    assemble_cleaned(final_train, train_features, "train"),
    assemble_cleaned(final_test, test_features, "test"),
]).sort_values(["CloseDate", "RecordId"]).reset_index(drop=True)

feature_columns = train_features.columns.tolist()
assert cleaned[feature_columns].isna().sum().sum() == 0
assert cleaned.loc[cleaned["split"] == "test", "CloseMonth"].nunique() == 1
assert cleaned.loc[cleaned["split"] == "test", "CloseMonth"].iloc[0] == str(test_month)
assert final_train["CloseMonth"].nunique() == selected_x

CLEANED_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(CLEANED_PATH, index=False, float_format="%.6f")

pd.DataFrame({
    "rows": cleaned.groupby("split").size(),
    "first_month": cleaned.groupby("split")["CloseMonth"].min(),
    "last_month": cleaned.groupby("split")["CloseMonth"].max(),
}).assign(feature_count=len(feature_columns), output_file=str(CLEANED_PATH))

,rows,first_month,last_month,feature_count,output_file
split,,,,,
test,12024,2026-05,2026-05,40,data/week3_cleaned.csv
train,381478,2023-05,2026-04,40,data/week3_cleaned.csv


In [7]:
cleaned.head()

,RecordId,CloseDate,CloseMonth,ClosePrice,split,LivingArea_missing,LivingArea_scaled,Bedrooms_missing,Bedrooms_scaled,Bathrooms_missing,Bathrooms_scaled,LotSizeSquareFeet_missing,LotSizeSquareFeet_scaled,YearBuilt_missing,YearBuilt_scaled,Stories_missing,Stories_scaled,GarageSpaces_missing,GarageSpaces_scaled,ParkingTotal_missing,ParkingTotal_scaled,Latitude_missing,Latitude_scaled,Longitude_missing,Longitude_scaled,AttachedGarage_missing,AttachedGarage,PrivatePool_missing,PrivatePool,View_missing,View,Waterfront_missing,Waterfront,NewConstruction_missing,NewConstruction,Fireplace_missing,Fireplace,City_missing,City_frequency,County_missing,County_frequency,PostalCode_missing,PostalCode_frequency,Levels_missing,Levels_frequency
0,149621,2023-05-01,2023-05,"695,960.0000",train,0,-0.7986,0,-0.5232,0,-0.5681,0,-0.2900,0,-0.7134,0,-0.6587,0,0.0622,0,-0.3715,0,-0.5784,0,0.2354,0,1,0,0,0,0,1,0,0,0,0,1,0,0.0022,0,0.2406,0,0.0011,0,0.5566
1,150236,2023-05-01,2023-05,"1,700,000.0000",train,0,-0.2269,0,0.5818,0,-0.5681,0,-0.2319,0,-0.5662,0,-0.6587,0,0.0622,0,-0.3715,0,-0.6595,0,0.4235,0,1,0,0,0,0,1,0,0,0,0,1,0,0.0031,0,0.0938,0,0.0017,0,0.5566
2,150421,2023-05-01,2023-05,"1,575,000.0000",train,0,0.4545,0,1.6868,0,0.4019,0,-0.2296,0,-2.4066,0,1.5181,0,0.0622,0,-0.3715,0,-0.4123,0,0.3100,0,0,0,0,0,1,1,0,0,0,0,1,0,0.0007,0,0.2406,0,0.0007,0,0.3013
3,150422,2023-05-01,2023-05,"2,675,000.0000",train,0,0.0690,0,0.5818,0,-0.5681,0,-0.1765,0,0.0963,1,-0.6587,0,1.3109,1,-0.3715,0,1.4676,0,-1.7411,0,1,1,0,1,1,1,0,0,0,0,1,0,0.0027,0,0.0437,0,0.0015,1,0.1039
4,150446,2023-05-01,2023-05,"1,900,000.0000",train,0,0.7714,0,0.5818,0,0.4019,0,-0.2980,0,-0.1245,0,1.5181,0,0.0622,0,-0.3715,0,-0.7721,0,0.5430,0,1,0,0,0,1,1,0,0,0,0,1,0,0.0019,0,0.0938,0,0.0014,0,0.3013


## Week 3 outcome

The exported CSV contains the selected rolling train window and the latest-month test set. All model features are numeric and complete; `ClosePrice` remains unscaled as the prediction target. `CloseDate`, `CloseMonth`, `RecordId`, and `split` are metadata and must not be used as model inputs without deliberate feature engineering.